# Loss Function Comparison Experiments

Which loss function is most effective for jaguar re-identification?

**Tested Losses:**
- **Standard Batching Group** (6 losses): ArcFace variants, SubCenter, Cross-Entropy, Focal
- **PK Sampling Group** (3 losses): Triplet (hard/semi-hard), ArcFace+Triplet

**Fixed Settings:**
- Backbone: MegaDescriptor-L-384
- Dataset: JID_Master_Dataset (segmented, deduplicated, closed-set split)
- Epochs: 50

**Evaluation Metrics:**
- mAP (sample-level)
- CMC@k (k = 1, 5, 10, 20)
- mAP and CMC@k for identities with >=9 total samples across train/val/test
- Identity-balanced mAP (macro-average across identities)

Results logged to Wandb project: `camera-trap-reidentification`, tags: `loss_comparison`

## Setup

In [ ]:
import sys
from pathlib import Path
import logging

# Add src to path
project_root = Path.cwd().parent / "camera-trap-footage"
sys.path.insert(0, str(project_root / "src"))

from jaguars.common.logging_utils import setup_logger
from jaguars.reidentification.config import get_default_config
from jaguars.reidentification.experiments import get_loss_experiments
from jaguars.reidentification.training.train import run_processing as run_training
from jaguars.reidentification.wandb_results import fetch_latest_metrics_for_experiments

logger = setup_logger("loss_experiments", level=logging.INFO)
print("✓ Imports successful")

✓ Imports successful


## Configuration

Configure base settings for all loss experiments.

In [13]:
# Get default configuration
config = get_default_config()

# Run metadata for clear WandB separation
RUN_BATCH = "hf_0226_segmented_deduplicated_v2_cached"
DATASET_TAG = "dataset:fiftyone_JID_HF_0226_Segmented_Deduplicated_Cached"
SOURCE_TAG = "source:fiftyone_local_cache"
BACKBONE_TAG = "backbone:hf-hub:timm/efficientnetv2_rw_m.agc_in1k"

# Prepared local dataset
LOCAL_FO_DATASET_NAME = "JID_HF_0226_Segmented_Deduplicated_Cached"

# Wandb settings
config.wandb.enabled = True
config.wandb.entity = "jaguars"
config.wandb.project = "camera-trap-reidentification"
config.wandb.tags = ["loss_comparison", BACKBONE_TAG, DATASET_TAG, SOURCE_TAG, f"run_batch:{RUN_BATCH}"]

# Dataset settings (prepared local FiftyOne cache)
config.dataset.source = "fiftyone"
config.dataset.fo_dataset_name = LOCAL_FO_DATASET_NAME
config.dataset.fo_split_field = "closed_set_split"
config.dataset.fo_patches_field = "sam3_segmentations"
config.dataset.fo_label_field = "ground_truth"
config.dataset.fo_embeddings_field = "embeddings_EfficientNetV2_RW_M"

# Backbone settings (fixed)
config.backbone.name = "hf-hub:timm/efficientnetv2_rw_m.agc_in1k"
config.backbone.pretrained = True
config.backbone.embedding_dim = 1536
config.backbone.input_size = 384

# Training settings
config.training.num_epochs = 50

print(f"✓ Base config loaded")
print(f"  Wandb project: {config.wandb.project}")
print(f"  Wandb tags: {config.wandb.tags}")
print(f"  Dataset source: {config.dataset.source}")
print(f"  FiftyOne dataset: {config.dataset.fo_dataset_name}")
print(f"  Backbone: {config.backbone.name}")
print(f"  Embedding field: {config.dataset.fo_embeddings_field}")
print(f"  Epochs: {config.training.num_epochs}")

✓ Base config loaded
  Wandb project: camera-trap-reidentification
  Wandb tags: ['loss_comparison', 'backbone:hf-hub:timm/efficientnetv2_rw_m.agc_in1k', 'dataset:fiftyone_JID_HF_0226_Segmented_Deduplicated_Cached', 'source:fiftyone_local_cache', 'run_batch:hf_0226_segmented_deduplicated_v2_cached']
  Dataset source: fiftyone
  FiftyOne dataset: JID_HF_0226_Segmented_Deduplicated_Cached
  Backbone: hf-hub:timm/efficientnetv2_rw_m.agc_in1k
  Embedding field: embeddings_EfficientNetV2_RW_M
  Epochs: 50


In [14]:
import fiftyone as fo

if not fo.dataset_exists(LOCAL_FO_DATASET_NAME):
    raise ValueError(
        f"Local FiftyOne dataset '{LOCAL_FO_DATASET_NAME}' not found. "
        "Run notebooks/prepare_fiftyone_cache.ipynb first."
    )

dataset = fo.load_dataset(LOCAL_FO_DATASET_NAME)
images_view = dataset.select_group_slices("image") if dataset.group_field else dataset
print(f"✓ Using prepared local FiftyOne dataset: {LOCAL_FO_DATASET_NAME}")
print(f"  Total samples: {len(dataset)}")
print(f"  Image samples: {len(images_view)}")
print(f"  Splits: {images_view.count_values(config.dataset.fo_split_field)}")

if f"{config.dataset.fo_patches_field}.detections.{config.dataset.fo_embeddings_field}" not in images_view.get_field_schema(flat=True):
    print(
        f"⚠ Cached embedding field not found at detection-level: "
        f"{config.dataset.fo_patches_field}.detections.{config.dataset.fo_embeddings_field}"
    )

ServerSelectionTimeoutError: localhost:45061: [Errno 111] Connection refused (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms), Timeout: 30s, Topology Description: <TopologyDescription id: 69a6ea6890613236503f570d, topology_type: Single, servers: [<ServerDescription ('localhost', 45061) server_type: Unknown, rtt: None, error=AutoReconnect('localhost:45061: [Errno 111] Connection refused (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms)')>]>

## Get Loss Experiments

In [ ]:
# Get loss experiments with our custom config
loss_experiments = get_loss_experiments(base_config=config)

# Ensure all runs carry the same dataset/run-batch tags and a unique prefix
for exp in loss_experiments:
    exp.base_config.wandb.tags = list(dict.fromkeys(exp.base_config.wandb.tags + [DATASET_TAG, SOURCE_TAG, f"run_batch:{RUN_BATCH}"]))
    run_base = exp.base_config.wandb.run_name or exp.name
    exp.base_config.wandb.run_name = f"{RUN_BATCH}__{run_base}"

# Explicit check: Sub-Center ArcFace must be included
subcenter_experiments = [e for e in loss_experiments if "subcenter_arcface" in e.name]
if not subcenter_experiments:
    raise ValueError("Sub-Center ArcFace experiment is missing from get_loss_experiments()")
print(f"✓ Sub-Center ArcFace is included: {[e.name for e in subcenter_experiments]}")

print(f"✓ {len(loss_experiments)} loss experiments configured:")
print(f"\nStandard Batching Losses:")
for exp in [e for e in loss_experiments if "standard_batching" in e.base_config.wandb.tags]:
    print(f"  - {exp.name}: {exp.description}")

print(f"\nPK Sampling Losses:")
for exp in [e for e in loss_experiments if "pk_sampling" in e.base_config.wandb.tags]:
    print(f"  - {exp.name}: {exp.description}")

✓ Sub-Center ArcFace is included: ['loss_subcenter_arcface']
✓ 9 loss experiments configured:

Standard Batching Losses:
  - loss_arcface: ArcFace standard margin (m=0.5, s=64)
  - loss_arcface_soft: ArcFace soft margin (m=0.3, s=64)
  - loss_arcface_hard: ArcFace hard margin (m=0.7, s=64)
  - loss_subcenter_arcface: SubCenter ArcFace (handles intra-class variation)
  - loss_cross_entropy: Cross-Entropy (classification baseline)
  - loss_focal: Focal Loss (handles class imbalance)

PK Sampling Losses:
  - loss_arcface_triplet_pk: ArcFace + Triplet (PK sampling, P=8, K=4)
  - loss_triplet_hard_pk: Triplet with hard mining (PK sampling, P=8, K=4)
  - loss_triplet_semi_hard_pk: Triplet with semi-hard mining (PK sampling, P=8, K=4)


## Run Standard Batching Experiments

ArcFace variants, SubCenter, Cross-Entropy, Focal (random batching)

In [ ]:
# Run standard batching experiments
standard_results = {}
standard_experiments = [e for e in loss_experiments if "standard_batching" in e.base_config.wandb.tags]

for experiment in standard_experiments:
    logger.info(f"Running experiment: {experiment.name}")
    logger.info(f"  Description: {experiment.description}")
    logger.info(f"  Tags: {experiment.base_config.wandb.tags}")
    logger.info(f"  Group: {experiment.group}")
    
    try:
        result = run_training(experiment.base_config)
        standard_results[experiment.name] = result
        logger.info(f"✓ {experiment.name} completed")
    except Exception as e:
        logger.error(f"✗ {experiment.name} failed: {e}")
        standard_results[experiment.name] = {"error": str(e)}

print(f"\n✓ All {len(standard_experiments)} standard batching experiments completed")

15:04:24 - jid_logger.loss_experiments - INFO - Running experiment: loss_arcface
15:04:24 - jid_logger.loss_experiments - INFO -   Description: ArcFace standard margin (m=0.5, s=64)
15:04:24 - jid_logger.loss_experiments - INFO -   Tags: ['loss_comparison', 'dataset:fiftyone_JID_HF_0226_Segmented_Deduplicated_Cached', 'source:fiftyone_local_cache', 'run_batch:hf_0226_segmented_deduplicated_v2_cached', 'standard_batching']


15:04:24 - jid_logger.loss_experiments - INFO -   Group: loss_comparison_standard
15:04:24 - jid_logger.reidentification.training - INFO - Starting re-identification training...
15:04:24 - jid_logger.reidentification.training - INFO - Dataset source: fiftyone
15:04:24 - jid_logger.reidentification.training - INFO - Backbone: hf-hub:BVRA/MegaDescriptor-L-384
15:04:24 - jid_logger.reidentification.training - INFO - Device: cuda
15:04:24 - jid_logger.reidentification.training - INFO - Resource validation passed


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /sc/home/philipp.kolbe/.netrc.
wandb: Currently logged in as: hpi-philipp-kolbe to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


15:04:27 - jid_logger.reidentification.training - INFO - Loading dataset...
15:04:58 - jid_logger.reidentification.training - INFO - Dataset loaded:
15:04:58 - jid_logger.reidentification.training - INFO -   Train: 1632 samples
15:04:58 - jid_logger.reidentification.training - INFO -   Val: 167 samples
15:04:58 - jid_logger.reidentification.training - INFO -   Num classes: 175
15:04:58 - jid_logger.reidentification.training - INFO - Using pre-computed embeddings
15:04:59 - jid_logger.reidentification.training - INFO - DataLoaders created:
15:04:59 - jid_logger.reidentification.training - INFO -   Train batches: 51
15:04:59 - jid_logger.reidentification.training - INFO -   Val batches: 6
Model initialized:
  Input dim: 1536
  Hidden dim: 512
  Embedding dim: 256
  Num classes: 175
  ArcFace margin: 0.5
  ArcFace scale: 64.0
  Total parameters: 964,608
15:04:59 - jid_logger.reidentification.training - INFO - Loss: arcface
15:04:59 - jid_logger.reidentification.training - INFO - Training 

15:05:01 - jid_logger.reidentification.training - INFO - Train Loss: 41.2254, Train Acc: 0.00%
15:05:01 - jid_logger.reidentification.training - INFO - Val Loss: 39.5159, Val Acc: 0.00%
15:05:01 - jid_logger.reidentification.training - INFO - Val mAP (identity-balanced): 0.0818
15:05:01 - jid_logger.reidentification.training - INFO - Val CMC@1: 0.1955, CMC@5: 0.2782
15:05:01 - jid_logger.reidentification.training - INFO - Learning Rate: 0.000100
15:05:01 - jid_logger.reidentification.training - INFO - Saved best model to data/models/reidentification/best_model.pt
15:05:01 - jid_logger.reidentification.training - INFO - 
Epoch 2/50


...

15:09:01 - jid_logger.reidentification.training - INFO - Saved best model to data/models/reidentification/best_model.pt
15:09:01 - jid_logger.reidentification.training - INFO - 
Epoch 49/50


15:09:01 - jid_logger.reidentification.training - INFO - Train Loss: 17.3390, Train Acc: 16.91%
15:09:01 - jid_logger.reidentification.training - INFO - Val Loss: 32.2462, Val Acc: 11.38%
15:09:01 - jid_logger.reidentification.training - INFO - Val mAP (identity-balanced): 0.1577
15:09:01 - jid_logger.reidentification.training - INFO - Val CMC@1: 0.2632, CMC@5: 0.4060
15:09:01 - jid_logger.reidentification.training - INFO - Learning Rate: 0.000050
15:09:01 - jid_logger.reidentification.training - INFO - 
Epoch 50/50


15:09:01 - jid_logger.reidentification.training - INFO - Train Loss: 17.1672, Train Acc: 18.14%
15:09:01 - jid_logger.reidentification.training - INFO - Val Loss: 32.3266, Val Acc: 11.38%
15:09:01 - jid_logger.reidentification.training - INFO - Val mAP (identity-balanced): 0.1607
15:09:01 - jid_logger.reidentification.training - INFO - Val CMC@1: 0.2707, CMC@5: 0.3985
15:09:01 - jid_logger.reidentification.training - INFO - Learning Rate: 0.000050
15:09:01 - jid_logger.reidentification.training - INFO - Saved checkpoint to data/models/reidentification/checkpoint_epoch_50.pt
15:09:01 - jid_logger.reidentification.training - INFO - ======================================================================
15:09:01 - jid_logger.reidentification.training - INFO - Training completed!
15:09:01 - jid_logger.reidentification.training - INFO - Best epoch: 48
15:09:01 - jid_logger.reidentification.training - INFO - Best val_map: 0.1638


epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇███
lr,███████████████████████████████████████▁
train/acc,▁▁▁▁▁▁▂▂▂▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▄▅▅▅▆▆▆▆▆▇▇▇▇█
train/batch_acc,▁▁▁▁▁▁▂▂▂▁▂▂▃▃▃▄▄▄▄▆▅▅▆▅▅▆▅▅▆▆▆▆▃▃▆▇▆▇▇█
train/batch_cls_loss,████▇▆▆▆▇▇▅▆▆▆▅▅▅▅▅▅▄▄▃▄▃▃▃▄▃▂▂▃▁▃▃▃▂▂▂▃
train/batch_loss,█▆▇▇▆▅▆▆▆▅▅▅▅▅▅▅▅▄▅▄▃▂▃▅▃▃▃▂▃▃▂▃▁▂▂▂▂▂▁▁
train/loss,█▇▇▇▇▆▆▆▆▅▅▅▅▅▅▄▄▄▄▄▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▁▁▁▁▁
val/acc,▁▁▁▁▃▄▄▄▄▄▅▄▅▅▅▅▅▅▅▆▆▆▆▇▇▆▇▇▇█▇█████████
val/cmc@1,▁▁▂▃▄▄▃▃▃▃▄▄▄▄▄▄▃▃▄▃▃▃▄▄▄▄▅▅▅▅▅▅▆▅▅▇▇▇█▇
val/cmc@10,▁▁▁▁▂▂▂▃▃▃▂▂▃▂▃▂▃▃▃▃▄▃▄▄▄▄▅▄▅▅▆▆▆▆▅█▆▇▇█
+10,...


15:09:03 - jid_logger.loss_experiments - INFO - ✓ loss_focal completed

✓ All 6 standard batching experiments completed


## Run PK Sampling Experiments

Triplet losses and combined ArcFace+Triplet (PK sampling with P=8, K=4)

In [ ]:
# Run PK sampling experiments
pk_results = {}
pk_experiments = [e for e in loss_experiments if "pk_sampling" in e.base_config.wandb.tags]

for experiment in pk_experiments:
    logger.info(f"Running experiment: {experiment.name}")
    logger.info(f"  Description: {experiment.description}")
    logger.info(f"  Tags: {experiment.base_config.wandb.tags}")
    logger.info(f"  Group: {experiment.group}")
    
    try:
        result = run_training(experiment.base_config)
        pk_results[experiment.name] = result
        logger.info(f"✓ {experiment.name} completed")
    except Exception as e:
        logger.error(f"✗ {experiment.name} failed: {e}")
        pk_results[experiment.name] = {"error": str(e)}

print(f"\n✓ All {len(pk_experiments)} PK sampling experiments completed")

15:09:03 - jid_logger.loss_experiments - INFO - Running experiment: loss_arcface_triplet_pk
15:09:03 - jid_logger.loss_experiments - INFO -   Description: ArcFace + Triplet (PK sampling, P=8, K=4)
15:09:03 - jid_logger.loss_experiments - INFO -   Tags: ['loss_comparison', 'dataset:fiftyone_JID_HF_0226_Segmented_Deduplicated_Cached', 'source:fiftyone_local_cache', 'run_batch:hf_0226_segmented_deduplicated_v2_cached', 'pk_sampling']
15:09:03 - jid_logger.loss_experiments - INFO -   Group: loss_comparison_pk
15:09:03 - jid_logger.reidentification.training - INFO - Starting re-identification training...
15:09:03 - jid_logger.reidentification.training - INFO - Dataset source: fiftyone
15:09:03 - jid_logger.reidentification.training - INFO - Backbone: hf-hub:BVRA/MegaDescriptor-L-384
15:09:03 - jid_logger.reidentification.training - INFO - Device: cuda
15:09:03 - jid_logger.reidentification.training - INFO - Resource validation passed


15:09:06 - jid_logger.reidentification.training - INFO - Loading dataset...
15:09:36 - jid_logger.reidentification.training - INFO - Dataset loaded:
15:09:36 - jid_logger.reidentification.training - INFO -   Train: 1632 samples
15:09:36 - jid_logger.reidentification.training - INFO -   Val: 167 samples
15:09:36 - jid_logger.reidentification.training - INFO -   Num classes: 175
15:09:36 - jid_logger.reidentification.training - INFO - Using pre-computed embeddings
15:09:36 - jid_logger.reidentification.training - INFO - Using PK Sampler: P=8, K=4
15:09:36 - jid_logger.reidentification.training - INFO -   Effective batch size: 32
15:09:36 - jid_logger.reidentification.training - INFO - DataLoaders created:
15:09:36 - jid_logger.reidentification.training - INFO -   Train batches: 10
15:09:36 - jid_logger.reidentification.training - INFO -   Val batches: 6
Model initialized:
  Input dim: 1536
  Hidden dim: 512
  Embedding dim: 256
  Num classes: 175
  ArcFace margin: 0.5
  ArcFace scale: 64

15:09:36 - jid_logger.reidentification.training - INFO - Train Loss: 47.6013, Train Acc: 0.00%
15:09:36 - jid_logger.reidentification.training - INFO - Val Loss: 46.4564, Val Acc: 0.00%
15:09:36 - jid_logger.reidentification.training - INFO - Val mAP (identity-balanced): 0.0976
15:09:36 - jid_logger.reidentification.training - INFO - Val CMC@1: 0.1880, CMC@5: 0.3233
15:09:36 - jid_logger.reidentification.training - INFO - Learning Rate: 0.000100
15:09:36 - jid_logger.reidentification.training - INFO - Saved best model to data/models/reidentification/best_model.pt
15:09:36 - jid_logger.reidentification.training - INFO - 
Epoch 2/50


15:09:37 - jid_logger.reidentification.training - INFO - Train Loss: 46.8492, Train Acc: 0.00%
15:09:37 - jid_logger.reidentification.training - INFO - Val Loss: 45.5954, Val Acc: 0.00%
15:09:37 - jid_logger.reidentification.training - INFO - Val mAP (identity-balanced): 0.0939
15:09:37 - jid_logger.reidentification.training - INFO - Val CMC@1: 0.1880, CMC@5: 0.3308
15:09:37 - jid_logger.reidentification.training - INFO - Learning Rate: 0.000100
15:09:37 - jid_logger.reidentification.training - INFO - 
Epoch 3/50


15:09:37 - jid_logger.reidentification.training - INFO - Train Loss: 46.1421, Train Acc: 0.00%
15:09:37 - jid_logger.reidentification.training - INFO - Val Loss: 45.1685, Val Acc: 0.00%
15:09:37 - jid_logger.reidentification.training - INFO - Val mAP (identity-balanced): 0.0921
15:09:37 - jid_logger.reidentification.training - INFO - Val CMC@1: 0.1880, CMC@5: 0.3008
15:09:37 - jid_logger.reidentification.training - INFO - Learning Rate: 0.000100
15:09:37 - jid_logger.reidentification.training - INFO - 
Epoch 4/50


...

15:10:51 - jid_logger.reidentification.training - INFO - Train Loss: 38.9455, Train Acc: 0.00%


15:10:51 - jid_logger.reidentification.training - INFO - Val Loss: 39.2854, Val Acc: 0.00%
15:10:51 - jid_logger.reidentification.training - INFO - Val mAP (identity-balanced): 0.0871
15:10:51 - jid_logger.reidentification.training - INFO - Val CMC@1: 0.1955, CMC@5: 0.2932
15:10:51 - jid_logger.reidentification.training - INFO - Learning Rate: 0.000100
15:10:51 - jid_logger.reidentification.training - INFO - Saved checkpoint to data/models/reidentification/checkpoint_epoch_10.pt
15:10:51 - jid_logger.reidentification.training - INFO - 
Epoch 11/50


15:10:52 - jid_logger.reidentification.training - INFO - Train Loss: 38.7074, Train Acc: 0.00%
15:10:52 - jid_logger.reidentification.training - INFO - Val Loss: 39.1985, Val Acc: 0.00%
15:10:52 - jid_logger.reidentification.training - INFO - Val mAP (identity-balanced): 0.0879
15:10:52 - jid_logger.reidentification.training - INFO - Val CMC@1: 0.2030, CMC@5: 0.2932
15:10:52 - jid_logger.reidentification.training - INFO - Learning Rate: 0.000100
15:10:52 - jid_logger.reidentification.training - INFO - Early stopping triggered after 11 epochs
15:10:52 - jid_logger.reidentification.training - INFO - ======================================================================
15:10:52 - jid_logger.reidentification.training - INFO - Training completed!
15:10:52 - jid_logger.reidentification.training - INFO - Best epoch: 1
15:10:52 - jid_logger.reidentification.training - INFO - Best val_map: 0.0931


epoch,▁▂▂▃▄▅▅▆▇▇█
lr,▁▁▁▁▁▁▁▁▁▁▁
train/acc,▁▁▁▁▁▁▁▁▁▁▁
train/batch_acc,▁▁▁▁▁▁▁▁▁▁▁
train/batch_cls_loss,▅██▇▆▃▄▃▅▁▁
train/batch_loss,▅██▇▆▂▄▃▅▁▁
train/batch_triplet_loss,▆▁▁▅▄▃▃▄▂█▃
train/loss,█▇▇▆▄▄▃▃▂▂▁
val/acc,▁▁▁▁▁▁▁▁▁▁▁
val/cmc@1,▅▃▃▆▁▅▃▆▃▆█
+11,...


15:10:53 - jid_logger.loss_experiments - INFO - ✓ loss_triplet_semi_hard_pk completed

✓ All 3 PK sampling experiments completed


## Summary

Display results from all loss experiments.

In [ ]:
# Print summary of latest W&B results
import pandas as pd

LOSS_NAME_MAP = {
    "loss_arcface": "ArcFace (m=0.5)",
    "loss_arcface_soft": "ArcFace (m=0.3)",
    "loss_arcface_hard": "ArcFace (m=0.7)",
    "loss_subcenter_arcface": "Sub-Center ArcFace",
    "loss_cross_entropy": "Cross-Entropy",
    "loss_focal": "Focal (γ=2.0)",
    "loss_arcface_triplet_pk": "ArcFace + Triplet (PK)",
    "loss_triplet_hard_pk": "Triplet Hard (PK)",
    "loss_triplet_semi_hard_pk": "Triplet Semi-Hard (PK)",
}

wandb_results = fetch_latest_metrics_for_experiments(
    experiments=loss_experiments,
    entity=config.wandb.entity,
    project=config.wandb.project,
    additional_tags=[BACKBONE_TAG, DATASET_TAG, SOURCE_TAG, f"run_batch:{RUN_BATCH}"],
)
pk_names = {e.name for e in loss_experiments if "pk_sampling" in e.base_config.wandb.tags}
summary_data = []

for exp in loss_experiments:
    exp_name = exp.name
    result = wandb_results.get(exp_name, {"error": "Missing W&B result"})
    display_name = LOSS_NAME_MAP.get(exp_name, exp_name.replace("loss_", "").replace("_", " ").title())
    sampling = "PK" if exp_name in pk_names else "Standard"
    if "error" in result:
        summary_data.append({
            "Loss": display_name,
            "mAP": "ERROR",
            "CMC@1": "ERROR",
            "mAP (>=9 total)": "ERROR",
            "Sampling": sampling,
        })
    else:
        map_val = result.get("map", "N/A")
        cmc1 = result.get("cmc@1", "N/A")
        map_9 = result.get("map_min_total_9", "N/A")

        map_str = f"{map_val:.4f}" if isinstance(map_val, (int, float)) else str(map_val)
        cmc1_str = f"{cmc1:.4f}" if isinstance(cmc1, (int, float)) else str(cmc1)
        map_9_str = f"{map_9:.4f}" if isinstance(map_9, (int, float)) else str(map_9)

        summary_data.append({
            "Loss": display_name,
            "mAP": map_str,
            "CMC@1": cmc1_str,
            "mAP (>=9 total)": map_9_str,
            "Sampling": sampling,
        })

summary_df = pd.DataFrame(summary_data)
print("\n=== Loss Comparison Results (Latest W&B Runs) ===")
print(summary_df.to_string(index=False))
print(f"\nView detailed results at: https://wandb.ai/{config.wandb.entity}/{config.wandb.project}")


=== Loss Comparison Results ===
                  Loss mAP CMC@1 mAP (>=9 total) Sampling
       ArcFace (m=0.5) N/A   N/A             N/A Standard
       ArcFace (m=0.3) N/A   N/A             N/A Standard
       ArcFace (m=0.7) N/A   N/A             N/A Standard
    Sub-Center ArcFace N/A   N/A             N/A Standard
         Cross-Entropy N/A   N/A             N/A Standard
         Focal (γ=2.0) N/A   N/A             N/A Standard
ArcFace + Triplet (PK) N/A   N/A             N/A       PK
     Triplet Hard (PK) N/A   N/A             N/A       PK
Triplet Semi-Hard (PK) N/A   N/A             N/A       PK

View detailed results at: https://wandb.ai/jaguars/camera-trap-reidentification


## Export for Report

Generate report-ready artifacts (CSV, LaTeX table, PNG figure).

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

LOSS_NAME_MAP = {
    "loss_arcface": "ArcFace (m=0.5)",
    "loss_arcface_soft": "ArcFace (m=0.3)",
    "loss_arcface_hard": "ArcFace (m=0.7)",
    "loss_subcenter_arcface": "Sub-Center ArcFace",
    "loss_cross_entropy": "Cross-Entropy",
    "loss_focal": "Focal (γ=2.0)",
    "loss_arcface_triplet_pk": "ArcFace + Triplet (PK)",
    "loss_triplet_hard_pk": "Triplet Hard (PK)",
    "loss_triplet_semi_hard_pk": "Triplet Semi-Hard (PK)",
}

# Build summary_df from latest W&B runs if this cell is run standalone
if "summary_df" not in globals():
    wandb_results = fetch_latest_metrics_for_experiments(
        experiments=loss_experiments,
        entity=config.wandb.entity,
        project=config.wandb.project,
        additional_tags=[BACKBONE_TAG, DATASET_TAG, SOURCE_TAG, f"run_batch:{RUN_BATCH}"],
    )
    pk_names = {e.name for e in loss_experiments if "pk_sampling" in e.base_config.wandb.tags}

    summary_data = []
    for exp in loss_experiments:
        exp_name = exp.name
        result = wandb_results.get(exp_name, {"error": "Missing W&B result"})
        display_name = LOSS_NAME_MAP.get(exp_name, exp_name.replace("loss_", "").replace("_", " ").title())
        sampling = "PK" if exp_name in pk_names else "Standard"
        if "error" in result:
            summary_data.append({
                "Loss": display_name,
                "mAP": "ERROR",
                "CMC@1": "ERROR",
                "mAP (>=9 total)": "ERROR",
                "Sampling": sampling,
            })
        else:
            map_val = result.get("map", "N/A")
            cmc1 = result.get("cmc@1", "N/A")
            map_9 = result.get("map_min_total_9", "N/A")
            summary_data.append({
                "Loss": display_name,
                "mAP": f"{map_val:.4f}" if isinstance(map_val, (int, float)) else str(map_val),
                "CMC@1": f"{cmc1:.4f}" if isinstance(cmc1, (int, float)) else str(cmc1),
                "mAP (>=9 total)": f"{map_9:.4f}" if isinstance(map_9, (int, float)) else str(map_9),
                "Sampling": sampling,
            })
    summary_df = pd.DataFrame(summary_data)

run_batch = globals().get("RUN_BATCH", "manual_run")
output_dir = Path("notebooks/data/results/figures/report") / run_batch / "loss"
output_dir.mkdir(parents=True, exist_ok=True)

# Save tabular artifacts
csv_path = output_dir / "loss_summary.csv"
tex_path = output_dir / "loss_summary.tex"
summary_df.to_csv(csv_path, index=False)
summary_df.to_latex(tex_path, index=False)

# Save figure
plot_df = summary_df.copy()
plot_df["mAP_numeric"] = pd.to_numeric(plot_df["mAP"], errors="coerce")
plot_df = plot_df.dropna(subset=["mAP_numeric"]).sort_values("mAP_numeric", ascending=False)

fig_path = output_dir / "loss_map_bar.png"
plt.figure(figsize=(12, max(4, 0.4 * len(plot_df))))
colors = ["#1f77b4" if s == "Standard" else "#ff7f0e" for s in plot_df["Sampling"]]
plt.barh(plot_df["Loss"], plot_df["mAP_numeric"], color=colors)
plt.gca().invert_yaxis()
plt.xlabel("mAP")
plt.title("Loss Comparison")
plt.tight_layout()
plt.savefig(fig_path, dpi=300)
plt.close()

print(f"Saved CSV: {csv_path}")
print(f"Saved LaTeX table: {tex_path}")
print(f"Saved figure: {fig_path}")

Saved CSV: notebooks/data/results/figures/report/hf_0226_segmented_deduplicated_v2_cached/loss/loss_summary.csv
Saved LaTeX table: notebooks/data/results/figures/report/hf_0226_segmented_deduplicated_v2_cached/loss/loss_summary.tex
Saved figure: notebooks/data/results/figures/report/hf_0226_segmented_deduplicated_v2_cached/loss/loss_map_bar.png
